## Visualization of best features - indicators of condition of patient - As per time series data 

### Let's visualize feature importance for quality assessment techniques within each stroke patient

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

In [4]:
df_final = pd.read_csv('raw_appended_data.csv')

In [15]:
import numpy as np
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import Conv1D, MaxPooling1D, Flatten, Dense


#X = df_final.drop(columns = ['condition', 'participant_id', 'trial_number', 'validity', 'side', 'movement_smoothness', 'side', 'side.1', 'Unnamed: 0'])
X = df_final.drop(columns = ['side', 'hand_acceleration'])
y = df_final['condition'].apply(lambda x: 1 if x == 'affected' else 0).values  # 1 = affected, 0 = unaffected


## Leave one out method 

- This is a special case of k-fold cross validation where the number of folds equals the number of samples (n)

In [36]:
import numpy as np
from sklearn.metrics import accuracy_score
from keras.models import Sequential
from keras.layers import Conv1D, MaxPooling1D, Flatten, Dense
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Assuming df_final is your dataframe
X = df_final.drop(columns=['side', 'hand_acceleration'])
y = df_final['condition'].apply(lambda x: 1 if x == 'affected' else 0).values


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score
from keras.models import Sequential
from keras.layers import Conv1D, MaxPooling1D, Flatten, Dense
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Assuming df_final is your dataframe
X = df_final.drop(columns=['side', 'hand_acceleration'])
y = df_final['condition'].apply(lambda x: 1 if x == 'affected' else 0).values

# Get unique participant IDs
participants = df_final['participant_id'].unique()

# Reshape the data for Conv1D (assuming time series data for each participant)
def reshape_data(X):
    return np.expand_dims(X, axis=-1)  # Adding the third dimension for Conv1D

# Initialize the model
def create_model(input_shape):
    model = Sequential()
    model.add(Conv1D(64, 3, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(2))
    model.add(Flatten())
    model.add(Dense(64, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Perform Leave-One-Out Cross-Validation (LOO-CV)
accuracies = []

X = X.drop(columns=['participant_id', 'trial_number', 'condition'])

for participant in participants:
    # Split the data: leave one participant out
    train_indices = df_final['participant_id'] != participant
    test_indices = df_final['participant_id'] == participant

    
    # Now, use the indices directly on the entire DataFrame
    X_train, X_test = X[train_indices], X[test_indices]
    y_train, y_test = y[train_indices], y[test_indices]

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    

    # Ensure all columns in X_train are numeric and handle any NaNs
    #X_train = X_train_scaled.apply(pd.to_numeric, errors='coerce').fillna(0)
    #X_test = X_test_scaled.apply(pd.to_numeric, errors='coerce').fillna(0)

    # Ensure target variable y is binary and numeric
    y_train = y_train.astype(np.float32)
    y_test = y_test.astype(np.float32)

    # Reshape data for Conv1D (time steps, features)
    X_train_reshaped = np.expand_dims(X_train, axis=-1)
    X_test_reshaped = np.expand_dims(X_test, axis=-1)

    # Create and train the model
    model = create_model(input_shape=(X_train_reshaped.shape[1], 1))
    model.fit(X_train_reshaped, y_train, epochs=10, batch_size=32, verbose=1)

    # Predict and evaluate the model
    y_pred = (model.predict(X_test_reshaped) > 0.5).astype(int)  # Convert probabilities to 0 or 1

    accuracies.append(accuracy_score(y_test, y_pred))

# Calculate and print the average accuracy across all folds
average_accuracy = np.mean(accuracies)
print(f'Average Accuracy: {average_accuracy:.4f}')


c:\Users\shrinidhi.velan\AppData\Local\anaconda\envs\AutoMQ\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
51556/51556 ━━━━━━━━━━━━━━━━━━━━ 182s 4ms/step - accuracy: 0.5410 - loss: 181029126144.0000
Epoch 2/10
51556/51556 ━━━━━━━━━━━━━━━━━━━━ 195s 4ms/step - accuracy: 0.7161 - loss: 9731460.0000
Epoch 3/10
51556/51556 ━━━━━━━━━━━━━━━━━━━━ 161s 3ms/step - accuracy: 0.7704 - loss: 36619024.0000
Epoch 4/10
51556/51556 ━━━━━━━━━━━━━━━━━━━━ 140s 3ms/step - accuracy: 0.8029 - loss: 18244302.0000
Epoch 5/10
51556/51556 ━━━━━━━━━━━━━━━━━━━━ 134s 3ms/step - accuracy: 0.8628 - loss: 3648796.7500
Epoch 6/10
51556/51556 ━━━━━━━━━━━━━━━━━━━━ 133s 3ms/step - accuracy: 0.7112 - loss: 7726204.5000
Epoch 7/10
51556/51556 ━━━━━━━━━━━━━━━━━━━━ 135s 3ms/step - accuracy: 0.8036 - loss: 9299495.0000
Epoch 8/10
51556/51556 ━━━━━━━━━━━━━━━━━━━━ 146s 3ms/step - accuracy: 0.5612 - loss: 152936352.0000
Epoch 9/10
51556/51556 ━━━━━━━━━━━━━━━━━━━━ 143s 3ms/step - accuracy: 0.5442 - loss: 0.6892
Epoch 10/10
51556/51556 ━━━━━━━━━━━━━━━━━━━━ 141s 3ms/step - accuracy: 0.5432 - loss: 0.6894
1971/1971 ━━━━━━━━━━━━

c:\Users\shrinidhi.velan\AppData\Local\anaconda\envs\AutoMQ\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
51231/51231 ━━━━━━━━━━━━━━━━━━━━ 140s 3ms/step - accuracy: 0.5189 - loss: 196901208064.0000
Epoch 2/10
51231/51231 ━━━━━━━━━━━━━━━━━━━━ 136s 3ms/step - accuracy: 0.5466 - loss: 0.6888
Epoch 3/10
51231/51231 ━━━━━━━━━━━━━━━━━━━━ 150s 3ms/step - accuracy: 0.5465 - loss: 0.6888
Epoch 4/10
51231/51231 ━━━━━━━━━━━━━━━━━━━━ 145s 3ms/step - accuracy: 0.5464 - loss: 0.6888
Epoch 5/10
51231/51231 ━━━━━━━━━━━━━━━━━━━━ 143s 3ms/step - accuracy: 0.5457 - loss: 0.6890
Epoch 6/10
51231/51231 ━━━━━━━━━━━━━━━━━━━━ 141s 3ms/step - accuracy: 0.5460 - loss: 0.6889
Epoch 7/10
51231/51231 ━━━━━━━━━━━━━━━━━━━━ 140s 3ms/step - accuracy: 0.5464 - loss: 0.6888
Epoch 8/10
51231/51231 ━━━━━━━━━━━━━━━━━━━━ 149s 3ms/step - accuracy: 0.5464 - loss: 0.6888
Epoch 9/10
51231/51231 ━━━━━━━━━━━━━━━━━━━━ 138s 3ms/step - accuracy: 0.5464 - loss: 0.6888
Epoch 10/10
51231/51231 ━━━━━━━━━━━━━━━━━━━━ 143s 3ms/step - accuracy: 0.5460 - loss: 0.6889
2295/2295 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step


c:\Users\shrinidhi.velan\AppData\Local\anaconda\envs\AutoMQ\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


...

51501/51501 ━━━━━━━━━━━━━━━━━━━━ 124s 2ms/step - accuracy: 0.5832 - loss: 181953282048.0000
Epoch 2/10
51501/51501 ━━━━━━━━━━━━━━━━━━━━ 121s 2ms/step - accuracy: 0.7548 - loss: 3478366.7500
Epoch 3/10
51501/51501 ━━━━━━━━━━━━━━━━━━━━ 122s 2ms/step - accuracy: 0.8175 - loss: 8258729.5000
Epoch 4/10
51501/51501 ━━━━━━━━━━━━━━━━━━━━ 125s 2ms/step - accuracy: 0.8541 - loss: 6955391.5000
Epoch 5/10
51501/51501 ━━━━━━━━━━━━━━━━━━━━ 123s 2ms/step - accuracy: 0.8509 - loss: 2520742.5000
Epoch 6/10
51501/51501 ━━━━━━━━━━━━━━━━━━━━ 134s 3ms/step - accuracy: 0.8598 - loss: 3123715.0000
Epoch 7/10
51501/51501 ━━━━━━━━━━━━━━━━━━━━ 124s 2ms/step - accuracy: 0.7017 - loss: 1676880.3750
Epoch 8/10
51501/51501 ━━━━━━━━━━━━━━━━━━━━ 122s 2ms/step - accuracy: 0.6136 - loss: 4185869.5000
Epoch 9/10
51501/51501 ━━━━━━━━━━━━━━━━━━━━ 121s 2ms/step - accuracy: 0.5437 - loss: 0.6893
Epoch 10/10
51501/51501 ━━━━━━━━━━━━━━━━━━━━ 120s 2ms/step - accuracy: 0.5437 - loss: 0.6893
2026/2026 ━━━━━━━━━━━━━━━━━━━━ 2s 863

c:\Users\shrinidhi.velan\AppData\Local\anaconda\envs\AutoMQ\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


51211/51211 ━━━━━━━━━━━━━━━━━━━━ 130s 2ms/step - accuracy: 0.5208 - loss: 210503106560.0000
Epoch 2/10
51211/51211 ━━━━━━━━━━━━━━━━━━━━ 129s 3ms/step - accuracy: 0.5437 - loss: 0.6893
Epoch 3/10
51211/51211 ━━━━━━━━━━━━━━━━━━━━ 145s 3ms/step - accuracy: 0.5439 - loss: 0.6893
Epoch 4/10
51211/51211 ━━━━━━━━━━━━━━━━━━━━ 155s 3ms/step - accuracy: 0.5436 - loss: 0.6893
Epoch 5/10
51211/51211 ━━━━━━━━━━━━━━━━━━━━ 152s 3ms/step - accuracy: 0.5433 - loss: 0.6894
Epoch 6/10
51211/51211 ━━━━━━━━━━━━━━━━━━━━ 144s 3ms/step - accuracy: 0.5432 - loss: 0.6894
Epoch 7/10
51211/51211 ━━━━━━━━━━━━━━━━━━━━ 125s 2ms/step - accuracy: 0.5437 - loss: 0.6893
Epoch 8/10
51211/51211 ━━━━━━━━━━━━━━━━━━━━ 128s 3ms/step - accuracy: 0.5439 - loss: 0.6893
Epoch 9/10
51211/51211 ━━━━━━━━━━━━━━━━━━━━ 159s 3ms/step - accuracy: 0.5443 - loss: 0.6892
Epoch 10/10
51211/51211 ━━━━━━━━━━━━━━━━━━━━ 153s 3ms/step - accuracy: 0.5441 - loss: 0.6893
2316/2316 ━━━━━━━━━━━━━━━━━━━━ 2s 881us/step


c:\Users\shrinidhi.velan\AppData\Local\anaconda\envs\AutoMQ\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
50678/50678 ━━━━━━━━━━━━━━━━━━━━ 162s 3ms/step - accuracy: 0.5251 - loss: 187737227264.0000
Epoch 2/10
50678/50678 ━━━━━━━━━━━━━━━━━━━━ 157s 3ms/step - accuracy: 0.5445 - loss: 0.6892
Epoch 3/10
50678/50678 ━━━━━━━━━━━━━━━━━━━━ 153s 3ms/step - accuracy: 0.5441 - loss: 0.6893
Epoch 4/10
50678/50678 ━━━━━━━━━━━━━━━━━━━━ 170s 3ms/step - accuracy: 0.5448 - loss: 0.6891
Epoch 5/10
50678/50678 ━━━━━━━━━━━━━━━━━━━━ 172s 3ms/step - accuracy: 0.5443 - loss: 0.6892
Epoch 6/10
50678/50678 ━━━━━━━━━━━━━━━━━━━━ 169s 3ms/step - accuracy: 0.5449 - loss: 0.6891
Epoch 7/10
50678/50678 ━━━━━━━━━━━━━━━━━━━━ 144s 3ms/step - accuracy: 0.5451 - loss: 0.6891
Epoch 8/10
50678/50678 ━━━━━━━━━━━━━━━━━━━━ 169s 3ms/step - accuracy: 0.5438 - loss: 0.6893
Epoch 9/10
30869/50678 ━━━━━━━━━━━━━━━━━━━━ 57s 3ms/step - accuracy: 0.5442 - loss: 0.6892